# Testes depois do treino: interação iterativa e negativo entre núcleos

**Pergunta:** com recortes e o negativo do Cellpose, o pipeline estaciona em Dice ~0,835 (o Cellpose dá 0,845). As perdas que mexem
no marcador (DMap) ou na saída (BCE) não mudaram a segmentação, e o erro que sobra é de precisão: núcleos vizinhos fundidos,
sobretudo nas regiões densas. Duas ideias mexem no que entra na camada de segmentação e podem ser testadas **sem retreinar**, com
os modelos dos *folds* já gravados:

| Ideia | O que muda |
|---|---|
| **Interação iterativa do ScribblePrompt** | o ScribblePrompt roda 2 ou 3 vezes em cada bloco; a partir da segunda, recebe os *logits* da passada anterior como `mask_input`, como na interação em passos com que ele foi treinado |
| **Negativo entre núcleos** | além do fundo do Cellpose (pixels a mais de 3 px de qualquer máscara), vira negativo todo pixel que tem, a até 1 ou 2 px, dois núcleos **diferentes** do Cellpose: linhas finas nas fronteiras entre vizinhos |

Cada uma das 37 imagens de treino é prevista pelo modelo do *fold* em que ela ficou de fora, com a mesma avaliação em blocos do
treino. A condição **base** é o pipeline exatamente como foi treinado e precisa reproduzir o resultado registrado. Cada condição é
avaliada nos limiares de saída 0,5, 0,6 e 0,7.

**Tudo é mostrado no notebook**; salvar é opcional (`SALVAR_RESULTADOS`). Use um ambiente com **GPU**.

## 0. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

## 1. Configuração (a mesma do experimento testado)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.data.load.preprocessed_dataset import NEGATIVE_KEY, load_preprocessed
from src.evaluation.figures import pick_cases, prediction_figure
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import save_run, summarize
from src.losses.loss_composer import LossComposer
from src.losses.terms import DiceTerm
from src.models.configs.marker_unet_config import UNET_CONFIG
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.models.networks.marker_unet import MarkerUNet
from src.pipeline.steps.inference.frozen_segmentation_step import FrozenSegmentationStep
from src.pipeline.steps.inference.marker_step import MarkerStep
from src.pipeline.training_pipeline import TrainingPipeline
from src.pipeline.tiled_pipeline import TiledPipeline
from src.training.checkpoints import fold_name, load_pipeline
from src.training.kfold import TrainConfig, final_epochs, run_kfold

# Experimento cujos modelos são testados: a configuração abaixo precisa ser a dele (a célula seguinte confere).
EXPERIMENT_NAME = "recortes_negativo"
OVERWRITE = False            # recusa gravar por cima de resultados ou modelos que já existem

RGBA_ALPHA = "mask"          # 4º canal da MarkerUNet: máscara do Cellpose ("prob": probabilidade)
TILE_SIZE = 256             # recorte do treino e bloco da avaliação (px da imagem original)
CROPS_PER_IMAGE = 4          # recortes sorteados de cada imagem por época
TILE_OVERLAP = 0.25          # sobreposição dos blocos na avaliação (fração do lado)
SP_INPUT_SIZE = (128, 128)   # entrada do ScribblePrompt (o recorte reduzido)
TAU = 0.5                    # limiar do canal positivo
NEGATIVE_RADIUS = 3              # None: sem negativo; n: negativo = pixels a mais de n px de qualquer máscara do Cellpose

CONFIG = TrainConfig(
    k=5, batch_size=16, max_epochs=200, patience=20, min_delta=0.0,
    lr=1e-4, grad_clip=1.0, threshold=0.5, seed=42, log_every=10,
    crop_size=TILE_SIZE, crops_per_image=CROPS_PER_IMAGE,
)
DESCRIPTION = {
    "perdas": "soft Dice (1,0), sem regularização do marcador",
    "rede_final": f"ScribblePrompt-UNet v1 congelado, scribble_mode=positive, tau={TAU}, input_size={SP_INPUT_SIZE}"
                  + ("" if NEGATIVE_RADIUS is None
                     else f", negativo = fundo do Cellpose (faixa livre de {NEGATIVE_RADIUS} px)"),
    "marker_net": "MarkerUNet (smp Unet, resnet34 ImageNet, 4 canais)",
    "rgba_alpha": RGBA_ALPHA,
    "aumentacao": "rot90 + flips, sorteada a cada época",
    "recortes": f"treino com {CROPS_PER_IMAGE} recortes de {TILE_SIZE}² por imagem por época (MarkerUNet sem redução); avaliação da imagem inteira em blocos de {TILE_SIZE}² com sobreposição {TILE_OVERLAP} e janela de Hann",
}

assert torch.cuda.is_available() or "google.colab" not in sys.modules, "No Colab, use um ambiente com GPU."
print("Device:", CONFIG.device)
print(json.dumps(DESCRIPTION, indent=2, ensure_ascii=False))

In [ ]:
SALVAR_RESULTADOS = False    # True: grava as tabelas em docs/estudo/resultados/testes_pos_treino_<experimento>/
LIMIARES = [0.5, 0.6, 0.7]
PASSADAS = [2, 3]
RAIOS_ENTRE_NUCLEOS = [1, 2]
IMAGENS_20X = {"TCGA-HE-7128-01Z-00-DX1", "TCGA-HE-7129-01Z-00-DX1", "TCGA-HE-7130-01Z-00-DX1"}

with open(os.path.join("docs", "estudo", "resultados", EXPERIMENT_NAME, "config.json"), encoding="utf-8") as f:
    REFERENCIA = json.load(f)
if REFERENCIA["descricao"] != DESCRIPTION:
    raise ValueError("A configuração deste notebook não é a do experimento testado; copie-a do notebook dele.")
print(f"Configuração de {EXPERIMENT_NAME} conferida.")

## 2. Onde estão os modelos

In [ ]:
# Modelos treinados: no Colab, no Google Drive de quem roda o notebook (sobrevive ao fim da sessão); fora dele,
# em ~/cell-fuzzy-models. Troque MODELS_ROOT para gravar em outro lugar.
if "google.colab" in sys.modules:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        MODELS_ROOT = "/content/drive/MyDrive/cell-fuzzy-models"
    except Exception as erro:
        print(f"Drive não montado ({erro}): os modelos ficam no disco da sessão e somem quando ela terminar.")
        MODELS_ROOT = "/content/cell-fuzzy-models"
else:
    MODELS_ROOT = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))
MODELS_DIR = os.path.join(MODELS_ROOT, EXPERIMENT_NAME)
print("Modelos em:", MODELS_DIR)

## 3. Dados de treino

In [ ]:
train_samples = load_preprocessed("train", alpha=RGBA_ALPHA, negative_radius=NEGATIVE_RADIUS)
TRAIN_IDS = sorted(train_samples)
print(f"Treino: {len(train_samples)} imagens (o teste não é carregado neste notebook)")

d = [binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])["dice"] for s in train_samples.values()]
print(f"Cellpose sozinho, treino: Dice {np.mean(d):.4f} ± {np.std(d, ddof=1):.4f}")

## 4. Rede final e montagem do experimento

In [ ]:
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_ROOT, "scribbleprompt"))
FINAL_NET = ScribblePromptingNetwork(
    checkpoint=checkpoint, device=CONFIG.device, scribble_mode="positive",
    positive_threshold=TAU, input_size=SP_INPUT_SIZE,
)


def factory():
    """Monta um experimento novo: MarkerUNet recém-inicializada, pipeline em blocos e perdas.

    O treino passa recortes de TILE_SIZE² (a MarkerUNet os vê sem redução); a avaliação passa a imagem
    inteira, que o TiledPipeline divide em blocos de TILE_SIZE² e cola de volta. O ScribblePrompt é
    congelado e não guarda estado, então a mesma instância serve a todos os folds.
    """
    marker_net = MarkerUNet(config=UNET_CONFIG)
    pipeline = TrainingPipeline([
        MarkerStep(model=marker_net, differentiable=True, target_size=TILE_SIZE, device=CONFIG.device),
        FrozenSegmentationStep(final_network=FINAL_NET, device=CONFIG.device,
                               negative_key=None if NEGATIVE_RADIUS is None else NEGATIVE_KEY),
    ])
    composer = LossComposer([DiceTerm()])
    return marker_net, TiledPipeline(pipeline, tile_size=TILE_SIZE, overlap=TILE_OVERLAP), composer

## 5. Modelos dos *folds* e negativo entre núcleos

Cada modelo traz, no `.json` ao lado, as imagens de validação do *fold*: é assim que cada imagem é prevista pelo modelo que não a
viu. O negativo entre núcleos é calculado a partir das instâncias do Cellpose: um pixel é fronteira se, num disco de raio `r`,
houver dois rótulos diferentes de núcleo.

In [ ]:
import time

import pandas as pd
from scipy import ndimage
from scipy.stats import wilcoxon

from src.data.load.preprocessed_dataset import PreprocessedDataset, collate_samples
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import provenance

FOLDS = []
for k in range(CONFIG.k):
    pipeline, meta = load_pipeline(factory, MODELS_DIR, fold_name(k))
    FOLDS.append((k, pipeline, meta["val_ids"]))
print({k: len(ids) for k, _, ids in FOLDS}, "imagens de validação por fold")


def disco(raio):
    y, x = np.ogrid[-raio:raio + 1, -raio:raio + 1]
    return x * x + y * y <= raio * raio


def fronteira_entre_nucleos(rotulos, raio):
    """1 onde, num disco de raio ``raio``, há dois rótulos diferentes de núcleo (0 é fundo)."""
    grande = np.iinfo(np.int64).max
    r = rotulos.astype(np.int64)
    maior = ndimage.maximum_filter(r, footprint=disco(raio))
    menor = ndimage.minimum_filter(np.where(r > 0, r, grande), footprint=disco(raio))
    return (maior > 0) & (menor < grande) & (maior != menor)


def com_negativo_entre_nucleos(amostras, raio):
    novas = {}
    for image_id, s in amostras.items():
        s = dict(s)
        s[NEGATIVE_KEY] = np.maximum(s[NEGATIVE_KEY], fronteira_entre_nucleos(s["cellpose_segmentation"], raio)).astype(np.float32)
        novas[image_id] = s
    return novas

## 6. Avaliação de cada condição

In [ ]:
def avaliar(condicao, amostras, passadas=1):
    FINAL_NET.n_interactions = passadas
    linhas = []
    for k, pipeline, val_ids in FOLDS:
        for image_id in val_ids:
            batch = collate_samples([PreprocessedDataset(amostras, ids=[image_id])[0]])
            batch = {c: (v.to(CONFIG.device) if torch.is_tensor(v) else v) for c, v in batch.items()}
            with torch.no_grad():
                prob = pipeline.run(batch, verbose=False)["segmentation"][0, 0].float().cpu().numpy()
            s = amostras[image_id]
            base = binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])
            for limiar in LIMIARES:
                linhas.append({"condicao": condicao, "limiar": limiar, "id": image_id, "fold": k,
                               "aumento_20x": image_id in IMAGENS_20X,
                               **binary_metrics(prob, s["ground_truth"], limiar),
                               **{f"cellpose_{m}": v for m, v in base.items()}})
    FINAL_NET.n_interactions = 1
    return linhas


inicio = time.time()
LINHAS = avaliar("base", train_samples)
for p in PASSADAS:
    LINHAS += avaliar(f"{p} passadas", train_samples, passadas=p)
for r in RAIOS_ENTRE_NUCLEOS:
    LINHAS += avaliar(f"negativo entre núcleos, {r} px", com_negativo_entre_nucleos(train_samples, r))
df = pd.DataFrame(LINHAS)
media = df[df.limiar == 0.5].groupby("condicao")["dice"].mean()
melhor_passadas = max(PASSADAS, key=lambda p: media[f"{p} passadas"])
melhor_raio = max(RAIOS_ENTRE_NUCLEOS, key=lambda r: media[f"negativo entre núcleos, {r} px"])
combo = f"{melhor_passadas} passadas + negativo entre núcleos, {melhor_raio} px"
LINHAS += avaliar(combo, com_negativo_entre_nucleos(train_samples, melhor_raio), passadas=melhor_passadas)
df = pd.DataFrame(LINHAS)
print(f"{(time.time() - inicio) / 60:.1f} min")

## 7. Resultados

Média das 37 imagens. `Δ base`: diferença pareada contra a condição base no mesmo limiar, com melhora/piora e o p do Wilcoxon;
`acima do Cellpose`: em quantas imagens a condição passa do Cellpose. A primeira linha (base, 0,5) precisa ficar perto do Dice
registrado na validação cruzada do experimento.

In [ ]:
ORDEM = list(dict.fromkeys(df.condicao))
base05 = df[(df.condicao == "base") & (df.limiar == 0.5)].set_index("id")
registro = pd.read_csv(os.path.join("docs", "estudo", "resultados", EXPERIMENT_NAME, "folds.csv"))
print(f"Dice registrado na validação cruzada de {EXPERIMENT_NAME}: {registro.dice.mean():.4f} | base agora: {base05.dice.mean():.4f}")

tabela = []
for condicao in ORDEM:
    for limiar in LIMIARES:
        s = df[(df.condicao == condicao) & (df.limiar == limiar)].set_index("id").loc[base05.index]
        ref = df[(df.condicao == "base") & (df.limiar == limiar)].set_index("id").loc[base05.index]
        d = s.dice - ref.dice
        p = wilcoxon(s.dice, ref.dice).pvalue if (d != 0).any() else float("nan")
        tabela.append({"condição": condicao, "limiar": limiar, "Dice": s.dice.mean(), "precisão": s.precision.mean(),
                       "revocação": s.recall.mean(), "massa": s.mass_ratio.mean(), "Δ base": d.mean(),
                       "melhora/piora": f"{int((d > 0).sum())}/{int((d < 0).sum())}", "p": p,
                       "acima do Cellpose": int((s.dice > s.cellpose_dice).sum()), "Dice 20×": s[s.aumento_20x].dice.mean()})
tabela = pd.DataFrame(tabela)
print(f"Cellpose: Dice {base05.cellpose_dice.mean():.4f}, precisão {base05.cellpose_precision.mean():.4f}, "
      f"revocação {base05.cellpose_recall.mean():.4f}")
pd.set_option("display.width", 220)
tabela.round(4)

### Como ler a tabela acima

- **Dice, precisão, revocação, massa:** médias das 37 imagens, cada uma prevista pelo modelo do *fold* em que ficou de fora. A
  massa é o previsto dividido pelo GT (acima de 1, o pipeline segmenta demais).
- **Δ base:** a diferença média, imagem por imagem, contra a condição base no **mesmo** limiar; positiva = a condição ganhou.
  **melhora/piora** conta as imagens; **p** é o teste de Wilcoxon pareado (abaixo de 0,05, a diferença dificilmente é acaso).
- **acima do Cellpose:** em quantas das 37 imagens a condição passa do Cellpose.
- **Dice 20×:** a média das três imagens em 20×, onde os núcleos são pequenos e densos e a fusão de vizinhos é mais comum.
- Uma condição que ganhe aqui ainda precisa ser confirmada **treinando** com ela: o modelo foi treinado sem ela.

## 8. Figura

Mapas de erro de um recorte de 300 × 300 px da imagem mais densa entre as piores (`TCGA-HE-7129`, em 20×; se ela não estiver entre
as imagens avaliadas, a pior da condição base), no limiar 0,5, em cada condição. Acerto em cinza, falso positivo em verde-água, falso negativo em violeta. Núcleos vizinhos fundidos aparecem como
pontes verde-água entre dois núcleos.

In [ ]:
COR_VP = np.array([0xB8, 0xB7, 0xB0]) / 255
COR_FP = np.array([0x1B, 0xAF, 0x7A]) / 255
COR_FN = np.array([0x4A, 0x3A, 0xA7]) / 255
AVALIADAS = {i for _, _, ids in FOLDS for i in ids}
FIG_ID = "TCGA-HE-7129-01Z-00-DX1" if "TCGA-HE-7129-01Z-00-DX1" in AVALIADAS else base05.dice.idxmin()
LADO = 300
fatia = (slice(350, 350 + LADO), slice(350, 350 + LADO))


def mapa_erro(pred, gt):
    out = np.ones(gt.shape + (3,))
    out[pred & gt] = COR_VP
    out[pred & ~gt] = COR_FP
    out[~pred & gt] = COR_FN
    return out


def prever(amostras, passadas):
    FINAL_NET.n_interactions = passadas
    k, pipeline, _ = next(f for f in FOLDS if FIG_ID in f[2])
    batch = collate_samples([PreprocessedDataset(amostras, ids=[FIG_ID])[0]])
    batch = {c: (v.to(CONFIG.device) if torch.is_tensor(v) else v) for c, v in batch.items()}
    with torch.no_grad():
        prob = pipeline.run(batch, verbose=False)["segmentation"][0, 0].float().cpu().numpy()
    FINAL_NET.n_interactions = 1
    return prob


variantes = {"base": (train_samples, 1)}
for p in PASSADAS:
    variantes[f"{p} passadas"] = (train_samples, p)
for r in RAIOS_ENTRE_NUCLEOS:
    variantes[f"negativo entre núcleos, {r} px"] = (com_negativo_entre_nucleos(train_samples, r), 1)
variantes[combo] = (com_negativo_entre_nucleos(train_samples, melhor_raio), melhor_passadas)

gt = train_samples[FIG_ID]["ground_truth"] > 0
fig, axes = plt.subplots(1, len(variantes) + 1, figsize=(3.2 * (len(variantes) + 1), 3.6))
axes[0].imshow(train_samples[FIG_ID]["image"][fatia])
axes[0].set_title("imagem", fontsize=9)
for ax, (nome, (amostras, passadas)) in zip(axes[1:], variantes.items()):
    prob = prever(amostras, passadas)
    d = binary_metrics(prob, gt)["dice"]
    ax.imshow(mapa_erro((prob >= 0.5)[fatia], gt[fatia]), interpolation="nearest")
    ax.set_title(f"{nome}\nDice {d:.3f}".replace(".", ","), fontsize=9)
for ax in axes:
    ax.set_xticks([]), ax.set_yticks([])
plt.tight_layout()
plt.show()

## 9. Salvar (opcional)

In [ ]:
if SALVAR_RESULTADOS:
    pasta = os.path.join("docs", "estudo", "resultados", f"testes_pos_treino_{EXPERIMENT_NAME}")
    os.makedirs(pasta, exist_ok=False)
    df.to_csv(os.path.join(pasta, "por_imagem.csv"), index=False)
    tabela.to_csv(os.path.join(pasta, "resumo.csv"), index=False)
    fig.savefig(os.path.join(pasta, "figura.png"), dpi=100, bbox_inches="tight")
    with open(os.path.join(pasta, "config.json"), "w", encoding="utf-8") as f:
        json.dump({"experimento": EXPERIMENT_NAME, "limiares": LIMIARES, "passadas": PASSADAS,
                   "raios_entre_nucleos": RAIOS_ENTRE_NUCLEOS, "procedencia": provenance()}, f, indent=2, ensure_ascii=False)
    print("Resultados em:", pasta)
else:
    print("Nada salvo (SALVAR_RESULTADOS = False).")